### Imports and data

In [1]:
import requests
from urllib.parse import urlencode

import pandas as pd
import numpy as np

from torch.utils.data import Dataset as torch_dataset, DataLoader
import torch
from torch.optim import AdamW
import torch.nn as nn

from transformers import DataCollatorWithPadding, AutoTokenizer, AutoModelForTokenClassification, BertPreTrainedModel, BertModel, AutoConfig, AutoModel
from typing import Dict, List, Union

from accelerate import Accelerator

from tqdm.auto import tqdm

from sklearn.metrics import f1_score

from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import numpy as np

import json
import os

In [2]:
import kagglehub

path = kagglehub.competition_download('neoai-2025-intent-detection-and-slot-filling')

print("Path to competition files:", path)

Path to competition files: C:\Users\raian\.cache\kagglehub\competitions\neoai-2025-intent-detection-and-slot-filling


In [3]:
# Read the dataset in conll format
def readConll(path):
    allSents = []
    allComments = []
    curSent = []
    curComments = []

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if curSent:
                    allSents.append(curSent)
                    allComments.append(curComments)
                    curSent = []
                    curComments = []
            else:
                if line[0] != '#':
                    tok = line.split('\t')
                    if tok[-1] == 'NoLabel':
                        tok[-1] = 'O'
                    curSent.append(tok)
                else:
                    curComments.append(line)

    if curSent:
        allSents.append(curSent)
        allComments.append(curComments)

    return allSents, allComments

In [4]:
data_dict = {
    'train': os.path.join(path, 'train.conll'),
    'val': os.path.join(path, 'validation.conll'),
    'test': os.path.join(path, 'test.conll')
}

parsed_data_dict = {}
for ds_type, filename in data_dict.items():
    # Read dataset
    allSents, allComments = readConll(filename)

    if ds_type == 'train':
        # Get all unique labels
        intents = []
        slots = []
        for sent in allSents:
            for word in sent:
                intents.append(word[2])
                slots.append(word[3])

        intents = list(set(intents))
        slots = list(set(slots))

        # Add slots that are not in the training set, but are present in the test and validation sets.
        slots.append('I-party_size_number')
        slots.append('I-condition_temperature')

        # Create labels dict
        id2label_intents = pd.Series(intents).to_dict()
        label2id_intents = {value: key for key, value in id2label_intents.items()}

        id2label_slots = pd.Series(slots).to_dict()
        label2id_slots = {value: key for key, value in id2label_slots.items()}

    # Save data
    parsed_data_dict[ds_type] = {'sents': allSents, 'comments': allComments}

In [5]:
parsed_data_dict['train']['sents'][0]

[['1', 'tell', 'weather/find', 'O'],
 ['2', 'me', 'weather/find', 'O'],
 ['3', 'the', 'weather/find', 'O'],
 ['4', 'weather', 'weather/find', 'O'],
 ['5', 'report', 'weather/find', 'O'],
 ['6', 'for', 'weather/find', 'O'],
 ['7', 'half', 'weather/find', 'B-location'],
 ['8', 'moon', 'weather/find', 'I-location'],
 ['9', 'bay', 'weather/find', 'I-location']]

In [6]:
class JointDataset(torch_dataset):
    def __init__(self, data, label2id_intents, label2id_slots):
        self.data = data
        self.label2id_intents = label2id_intents
        self.label2id_slots = label2id_slots

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]

        # Extracting tokens and labels
        tokens = [token[1] for token in example]
        slot_labels = [token[3] for token in example]
        if example[0][2] in self.label2id_intents:
          intent = self.label2id_intents[example[0][2]]
        else:
          intent = 0 # For test dataset

        encoded_slot_labels = []
        for slot in slot_labels:
            encoded_slot_labels.append(self.label2id_slots[slot])

        return {'input_text': tokens, 'intent_label': intent, 'slot_labels': encoded_slot_labels}

In [8]:
class JointDataCollator(DataCollatorWithPadding):
    def __init__(self, tokenizer, slot_pad_token_id=-100):
        super().__init__(tokenizer)
        self.slot_pad_token_id = slot_pad_token_id

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Extracting texts for tokenization
        texts = [f['input_text'] for f in features]

        # Tokenizing text splitted on words
        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            is_split_into_words=True,
            return_tensors="pt"
        )

        # Add the res part of fields
        batch['intent_labels'] = torch.tensor([f['intent_label'] for f in features])

        # Supplement slot labels to the maximum length in the batch
        max_length = batch['input_ids'].shape[1]
        slot_labels = []
        for i, f in enumerate(features):
            # labels = f['slot_labels']
            # padded = labels[:max_length] + [self.slot_pad_token_id] * (max_length - len(labels))
            # slot_labels.append(padded)

            labels = f['slot_labels']
            word_ids = batch.word_ids(batch_index=i)   # one entry per token
            aligned = []
            prev_word = None
            for word_id in word_ids:
                if word_id is None:                       # [CLS], [SEP], padding
                    aligned.append(self.slot_pad_token_id)
                elif word_id != prev_word:               # first subtoken of word
                    aligned.append(labels[word_id])
                else:                                    # continuation subtoken
                    aligned.append(self.slot_pad_token_id)
                prev_word = word_id
            slot_labels.append(aligned)

        batch['slot_labels'] = torch.tensor(slot_labels)

        return batch

### Model

In [9]:
class JointBertModel(BertPreTrainedModel):
    def __init__(self, config, num_intent_labels, num_slot_labels):
        super().__init__(config)

        self.num_intent_labels = num_intent_labels
        self.num_slot_labels = num_slot_labels

        self.bert = AutoModel.from_config(config)

        self.slot_classifier = nn.Linear(config.hidden_size, num_slot_labels + 1)
        self.intent_classifier = nn.Linear(config.hidden_size, num_intent_labels + 1)

        self.post_init()  # important for HF models

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                slot_labels=None, intent_labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        sequence_output = outputs.last_hidden_state

        slot_logits = self.slot_classifier(sequence_output)

        pooled_output = sequence_output[:, 0, :]
        intent_logits = self.intent_classifier(pooled_output)

        loss = None
        if slot_labels is not None and intent_labels is not None:
            loss_fct = nn.CrossEntropyLoss()

            slot_loss = loss_fct(
                slot_logits.view(-1, self.slot_classifier.out_features),
                slot_labels.view(-1)
            )
            intent_loss = loss_fct(
                intent_logits.view(-1, self.intent_classifier.out_features),
                intent_labels.view(-1)
            )

            loss = slot_loss * 3 + intent_loss

        return {
            "loss": loss,
            "slot_logits": slot_logits,
            "intent_logits": intent_logits
        }

In [14]:
#DO NOT CHANGE THIS CELL

# Model initialization
model_name = "Ilseyar-kfu/base_model"

config = AutoConfig.from_pretrained(model_name)

model = JointBertModel.from_pretrained(
    model_name,
    config=config,
    num_intent_labels=len(intents),
    num_slot_labels=len(slots)
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

optimizer = AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] JointBertModel LOAD REPORT from: Ilseyar-kfu/base_model
Key                      | Status  | 
-------------------------+---------+-
slot_classifier.weight   | MISSING | 
slot_classifier.bias     | MISSING | 
intent_classifier.weight | MISSING | 
intent_classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# DO NOT CHANGE THIS CODE!!!
SEED = 23

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Training

In [15]:
collator = JointDataCollator(tokenizer)

train_data = parsed_data_dict['train']['sents']
train_ds = JointDataset(train_data, label2id_intents, label2id_slots)
train_loader = DataLoader(train_ds, batch_size=32, collate_fn=collator, shuffle=True)

val_data = parsed_data_dict['val']['sents']
val_ds = JointDataset(val_data, label2id_intents, label2id_slots)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collator, shuffle=False)

test_data = parsed_data_dict['test']['sents']
test_ds = JointDataset(test_data, label2id_intents, label2id_slots)
test_loader = DataLoader(test_ds, batch_size=32, collate_fn=collator, shuffle=False)

accelerator = Accelerator()
model, optimizer, train_loader, val_loader, test_loader = accelerator.prepare(model, optimizer, train_loader, val_loader, test_loader)

In [16]:
def calculate_metrics(intent_preds, slot_preds, intent_labels, slot_labels):
    """
    Calculate f1-score for intent и slot classification
    """
    metrics = {}
    intent_preds, slot_preds, intent_labels, slot_labels = np.array(intent_preds), np.array(slot_preds), np.array(intent_labels), np.array(slot_labels)

    metrics['intent_weighted_f1'] = f1_score(intent_labels, intent_preds, average='weighted')

    # Filter paddings
    mask = slot_labels != -100
    flat_slot_preds = slot_preds[mask]
    flat_slot_true = slot_labels[mask]

    metrics['slot_weighted_f1'] = f1_score(flat_slot_true, flat_slot_preds, average='weighted')

    metrics['avg_weighted_f1'] = (metrics['intent_weighted_f1'] + metrics['slot_weighted_f1']) / 2

    return metrics

In [17]:
from transformers import get_linear_schedule_with_warmup

num_train_epochs = 3
lerning_rate = 2e-5
sheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = 1*len(train_loader), num_training_steps=num_train_epochs*len(train_loader))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    train_loss = 0
    intent_trues_epoch, intent_preds_epoch = [], []
    slot_trues_epoch, slot_preds_epoch = [], []
    for batch in tqdm(train_loader):
        outputs = model(**batch)
        loss = outputs['loss']
        train_loss += (loss.item() / batch['input_ids'].shape[0])
        accelerator.backward(loss)

        optimizer.step()
        sheduler.step()
        optimizer.zero_grad()

        # Save results
        intent_preds = torch.argmax(outputs['intent_logits'], dim=1).cpu().numpy()
        intent_preds_epoch.extend(intent_preds)
        intent_true = batch['intent_labels'].cpu().numpy()
        intent_trues_epoch.extend(intent_true)
        slot_preds = torch.argmax(outputs['slot_logits'], dim=2).cpu().numpy()
        slot_preds_epoch.extend(slot_preds.reshape(-1).tolist())
        slot_true = batch['slot_labels'].cpu().numpy()
        slot_trues_epoch.extend(slot_true.reshape(-1).tolist())

    # Calculate Metrics
    train_metrics = calculate_metrics(intent_preds_epoch, slot_preds_epoch, intent_trues_epoch, slot_trues_epoch)

    # Evaluation
    train_loss /= len(train_loader)
    val_loss = 0
    intent_trues_epoch, intent_preds_epoch = [], []
    slot_trues_epoch, slot_preds_epoch = [], []
    model.eval()
    for batch in val_loader:
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs['loss']
            val_loss += (loss.item() / batch['input_ids'].shape[0])

            # save results
            intent_preds = torch.argmax(outputs['intent_logits'], dim=1).cpu().numpy()
            intent_preds_epoch.extend(intent_preds)
            intent_true = batch['intent_labels'].cpu().numpy()
            intent_trues_epoch.extend(intent_true)
            slot_preds = torch.argmax(outputs['slot_logits'], dim=2).cpu().numpy()
            slot_preds_epoch.extend(slot_preds.reshape(-1).tolist())
            slot_true = batch['slot_labels'].cpu().numpy()
            slot_trues_epoch.extend(slot_true.reshape(-1).tolist())

    # calculate metrics
    val_metrics = calculate_metrics(intent_preds_epoch, slot_preds_epoch, intent_trues_epoch, slot_trues_epoch)

    val_loss /= len(val_loader)
 
    print(f'Epoch {epoch}:')
    print(f'Train Loss: {train_loss} Val Loss: {val_loss}')
    print(f'Train metrics: {train_metrics}')
    print(f'Val metrics: {val_metrics}')
    print()

  0%|          | 0/1163 [00:00<?, ?it/s]

Epoch 0:
Train Loss: 0.14060019222844908 Val Loss: 0.34768090559088666
Train metrics: {'intent_weighted_f1': 0.7181511596967155, 'slot_weighted_f1': 0.693042036341485, 'avg_weighted_f1': 0.7055965980191002}
Val metrics: {'intent_weighted_f1': 0.24514977369246815, 'slot_weighted_f1': 0.42073635681449145, 'avg_weighted_f1': 0.33294306525347983}



  0%|          | 0/1163 [00:00<?, ?it/s]

KeyboardInterrupt: 